In [1]:
# 06e-1. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

STATE_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
    / "state_dataset.parquet"
)

SUPERVISED_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
    / "supervised_dataset.parquet"
)


print(
    "state:",
    STATE_DATASET_PATH.exists()
)

print(
    "supervised:",
    SUPERVISED_DATASET_PATH.exists()
)

state: True
supervised: True


In [2]:
# 06e-2. dataset 불러오기

state_dataset = (
    pq.read_table(
        STATE_DATASET_PATH
    )
    .to_pandas()
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)

supervised_dataset = (
    pq.read_table(
        SUPERVISED_DATASET_PATH
    )
    .to_pandas()
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "state:",
    state_dataset.shape
)

print(
    "state signals:",
    state_dataset[
        "signal_date"
    ].nunique()
)

print(
    "supervised:",
    supervised_dataset.shape
)

print(
    "supervised signals:",
    supervised_dataset[
        "signal_date"
    ].nunique()
)

state: (21900, 34)
state signals: 438
supervised: (21779, 39)
supervised signals: 436


In [3]:
# 06e-3. state universe 크기 확인

state_universe_size = (
    state_dataset
    .groupby(
        "signal_date"
    )[
        "ticker"
    ]
    .nunique()
)


print(
    state_universe_size.describe()
)

print(
    "all 50:",
    (
        state_universe_size
        == 50
    ).all()
)

count    438.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
Name: ticker, dtype: float64
all 50: True


In [4]:
# 06e-4. baseline feature 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]


BASELINE_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)

TARGET = "target_return"


print(
    "baseline features:",
    len(BASELINE_FEATURES)
)

baseline features: 28


rank_scaled=2×N+1rank−1

절대 단위를 없애고 오늘 50종목 중 상대적으로 높은가/낮은가?만 남

In [6]:
# 06e-4. baseline feature 설정

ASSET_FEATURES = [
    
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]


BASELINE_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)

TARGET = "target_return"


print(
    "baseline features:",
    len(BASELINE_FEATURES)
)

baseline features: 28


In [7]:
# 06e-5. cross-sectional rank 생성

rank_dataset = (
    state_dataset[
        [
            "signal_date",
            "ticker"
        ]
        + ASSET_FEATURES
    ]
    .copy()
)


RANK_FEATURES = []


for feature in ASSET_FEATURES:

    rank_column = (
        f"{feature}_rank"
    )

    ranks = (
        rank_dataset
        .groupby(
            "signal_date"
        )[feature]
        .rank(
            method="average"
        )
    )

    counts = (
        rank_dataset
        .groupby(
            "signal_date"
        )[feature]
        .transform(
            "count"
        )
    )

    rank_dataset[
        rank_column
    ] = (
        2
        * ranks
        / (
            counts
            + 1
        )
        - 1
    )

    RANK_FEATURES.append(
        rank_column
    )


print(
    "rank features:",
    len(RANK_FEATURES)
)

print(
    RANK_FEATURES
)

rank features: 9
['return_1d_rank', 'return_5d_rank', 'momentum_20d_rank', 'momentum_60d_rank', 'volatility_20d_rank', 'drawdown_20d_rank', 'trading_value_ma20_rank', 'trading_value_ratio_20d_rank', 'log_market_cap_rank']


In [8]:
# 06e-6. rank 품질 확인

print(
    "rank nan:",
    rank_dataset[
        RANK_FEATURES
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "rank inf:",
    np.isinf(
        rank_dataset[
            RANK_FEATURES
        ]
    )
    .sum()
    .sum()
)

print(
    "rank min:",
    rank_dataset[
        RANK_FEATURES
    ]
    .min()
    .min()
)

print(
    "rank max:",
    rank_dataset[
        RANK_FEATURES
    ]
    .max()
    .max()
)

rank nan: 0
rank inf: 0
rank min: -0.9607843137254902
rank max: 0.9607843137254901


In [9]:
# 06e-7. rank feature merge

rank_features_table = (
    rank_dataset[
        [
            "signal_date",
            "ticker"
        ]
        + RANK_FEATURES
    ]
    .copy()
)


supervised_rank_dataset = (
    supervised_dataset
    .merge(
        rank_features_table,
        on=[
            "signal_date",
            "ticker"
        ],
        how="left",
        validate="one_to_one"
    )
)


print(
    "shape:",
    supervised_rank_dataset.shape
)

print(
    "signals:",
    supervised_rank_dataset[
        "signal_date"
    ].nunique()
)

print(
    "rank nan:",
    supervised_rank_dataset[
        RANK_FEATURES
    ]
    .isna()
    .sum()
    .sum()
)

shape: (21779, 48)
signals: 436
rank nan: 0


In [10]:
# 06e-8. feature experiment 설정

EXPERIMENT_FEATURES = (
    BASELINE_FEATURES
    + RANK_FEATURES
)


print(
    "baseline:",
    len(BASELINE_FEATURES)
)

print(
    "rank added:",
    len(RANK_FEATURES)
)

print(
    "experiment:",
    len(EXPERIMENT_FEATURES)
)

baseline: 28
rank added: 9
experiment: 37


In [ ]:
Ridge baseline 28개 vs 동일 Ridge + rank 9개 = 37개만 비교.
Gu·Kelly·Xiu 논문처럼 cross-sectional characteristics를 활용하는 접근이
자산수익률 예측에서 쓰였다는 문헌적 근거는 있지만,
그게 KRX 데이터에서도 개선된다는 보장은 없기 때문에
동일한 OOS protocol로 직접 확인하는 것임.

Ridge에서는 raw feature와 rank feature의 단위가 다르기 때문에
standardscaler → ridge를 pipeline 안에서 유지함. 
StandardScaler는 train에서 평균·표준편차를 학습하고 이후 데이터에 transform하며,
pipeline을 쓰면 preprocessing을 test에 먼저 fit하는 leakage를 막을 수 있을듯

In [12]:
# 06e-9. ridge experiment 설정

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


RIDGE_ALPHAS = np.logspace(
    -4,
    8,
    13
)


print(
    RIDGE_ALPHAS
)

[1.e-04 1.e-03 1.e-02 1.e-01 1.e+00 1.e+01 1.e+02 1.e+03 1.e+04 1.e+05
 1.e+06 1.e+07 1.e+08]


In [13]:
# 06e-10. cross-sectional ic 함수

def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        if (
            group[prediction_column].nunique() < 2
            or
            group[target_column].nunique() < 2
        ):
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )

In [14]:
# 06e-11. walk-forward fold 생성

TRAIN_YEARS = 3
VALIDATION_MONTHS = 6
TEST_MONTHS = 6
STEP_MONTHS = 6


signal_start = (
    supervised_rank_dataset[
        "signal_date"
    ].min()
)

signal_end = (
    supervised_rank_dataset[
        "signal_date"
    ].max()
)


folds = []

validation_start = (
    signal_start
    + pd.DateOffset(
        years=TRAIN_YEARS
    )
)

fold_id = 1


while True:

    test_start = (
        validation_start
        + pd.DateOffset(
            months=VALIDATION_MONTHS
        )
    )

    test_end = (
        test_start
        + pd.DateOffset(
            months=TEST_MONTHS
        )
    )

    if test_end > signal_end:
        break

    folds.append(
        {
            "fold": fold_id,
            "train_start": signal_start,
            "train_end": validation_start,
            "validation_start":
                validation_start,
            "validation_end":
                test_start,
            "test_start":
                test_start,
            "test_end":
                test_end
        }
    )

    validation_start = (
        validation_start
        + pd.DateOffset(
            months=STEP_MONTHS
        )
    )

    fold_id += 1


print(
    "folds:",
    len(folds)
)

print(
    pd.DataFrame(
        folds
    ).to_string(
        index=False
    )
)

folds: 9
 fold train_start  train_end validation_start validation_end test_start   test_end
    1  2018-05-04 2021-05-04       2021-05-04     2021-11-04 2021-11-04 2022-05-04
    2  2018-05-04 2021-11-04       2021-11-04     2022-05-04 2022-05-04 2022-11-04
    3  2018-05-04 2022-05-04       2022-05-04     2022-11-04 2022-11-04 2023-05-04
    4  2018-05-04 2022-11-04       2022-11-04     2023-05-04 2023-05-04 2023-11-04
    5  2018-05-04 2023-05-04       2023-05-04     2023-11-04 2023-11-04 2024-05-04
    6  2018-05-04 2023-11-04       2023-11-04     2024-05-04 2024-05-04 2024-11-04
    7  2018-05-04 2024-05-04       2024-05-04     2024-11-04 2024-11-04 2025-05-04
    8  2018-05-04 2024-11-04       2024-11-04     2025-05-04 2025-05-04 2025-11-04
    9  2018-05-04 2025-05-04       2025-05-04     2025-11-04 2025-11-04 2026-05-04


In [15]:
# 06e-12. ridge validation 함수

def evaluate_ridge_alphas(
    train_df,
    validation_df,
    features
):

    X_train = train_df[
        features
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        features
    ]

    y_validation = validation_df[
        TARGET
    ]


    results = []


    for alpha in RIDGE_ALPHAS:

        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=alpha
                    )
                )
            ]
        )


        model.fit(
            X_train,
            y_train
        )


        prediction = model.predict(
            X_validation
        )


        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )


        results.append(
            {
                "alpha": alpha,
                "rmse": rmse
            }
        )


    return pd.DataFrame(
        results
    )

In [17]:
# 06e-13. rank-added ridge walk-forward

ridge_rank_fold_rows = []
ridge_rank_oos_frames = []


for fold in folds:

    train_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["train_end"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            >= fold[
                "validation_start"
            ]
        )
        &
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold[
                "validation_end"
            ]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold[
                "validation_end"
            ]
        )
    )

    test_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            >= fold["test_start"]
        )
        &
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["test_end"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["test_end"]
        )
    )


    train_df = (
        supervised_rank_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_rank_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_rank_dataset.loc[
            test_mask
        ]
        .copy()
    )


    validation_results = (
        evaluate_ridge_alphas(
            train_df,
            validation_df,
            EXPERIMENT_FEATURES
        )
    )


    best_row = (
        validation_results
        .sort_values(
            "rmse"
        )
        .iloc[0]
    )


    best_alpha = float(
        best_row[
            "alpha"
        ]
    )


    train_validation_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["test_start"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["test_start"]
        )
    )


    train_validation_df = (
        supervised_rank_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )


    model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=best_alpha
                )
            )
        ]
    )


    model.fit(
        train_validation_df[
            EXPERIMENT_FEATURES
        ],
        train_validation_df[
            TARGET
        ]
    )


    prediction = model.predict(
        test_df[
            EXPERIMENT_FEATURES
        ]
    )


    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )


    test_result[
        "prediction"
    ] = prediction

    test_result[
        "fold"
    ] = fold["fold"]


    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )


    rmse = np.sqrt(
        mean_squared_error(
            test_result[
                TARGET
            ],
            prediction
        )
    )

    mae = mean_absolute_error(
        test_result[
            TARGET
        ],
        prediction
    )

    r2 = r2_score(
        test_result[
            TARGET
        ],
        prediction
    )


    ridge_rank_fold_rows.append(
        {
            "fold":
                fold["fold"],
            "best_alpha":
                best_alpha,
            "validation_rmse":
                best_row["rmse"],
            "test_rows":
                len(test_df),
            "test_signals":
                test_df[
                    "signal_date"
                ].nunique(),
            "rmse":
                rmse,
            "mae":
                mae,
            "r2":
                r2,
            "mean_ic": (
                ic_values.mean()
                if len(ic_values) > 0
                else np.nan
            ),
            "median_ic": (
                np.median(
                    ic_values
                )
                if len(ic_values) > 0
                else np.nan
            ),
            "ic_signals":
                len(ic_values)
        }
    )


    ridge_rank_oos_frames.append(
        test_result
    )


    print(
        "fold",
        fold["fold"],
        "done"
    )

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
fold 6 done
fold 7 done
fold 8 done
fold 9 done


In [18]:
# 06e-14. rank-added ridge 결과

ridge_rank_fold_results = (
    pd.DataFrame(
        ridge_rank_fold_rows
    )
)


ridge_rank_oos = (
    pd.concat(
        ridge_rank_oos_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    ridge_rank_fold_results.to_string(
        index=False
    )
)


print(
    "\noos shape:",
    ridge_rank_oos.shape
)

print(
    "signals:",
    ridge_rank_oos[
        "signal_date"
    ].nunique()
)

 fold  best_alpha  validation_rmse  test_rows  test_signals     rmse      mae        r2   mean_ic  median_ic  ic_signals
    1    100000.0         0.065538       1250            25 0.064158 0.047195 -0.012893 -0.037030  -0.042497          25
    2     10000.0         0.063468       1250            25 0.065214 0.047422 -0.017993 -0.065830  -0.072557          25
    3    100000.0         0.064831       1248            25 0.074759 0.049252 -0.016429 -0.017058  -0.043361          25
    4 100000000.0         0.074470       1250            25 0.076923 0.050796 -0.008782 -0.072369  -0.049796          25
    5 100000000.0         0.076923       1197            24 0.076069 0.051544 -0.005063 -0.092528  -0.120144          24
    6 100000000.0         0.076069       1249            25 0.081828 0.057120 -0.000333 -0.083479  -0.090228          25
    7   1000000.0         0.081827       1198            24 0.084040 0.062015 -0.000098 -0.048805  -0.024058          24
    8 100000000.0         0.0840

In [19]:
# 06e-15. rank-added ridge oos 성능

rank_rmse = np.sqrt(
    mean_squared_error(
        ridge_rank_oos[
            TARGET
        ],
        ridge_rank_oos[
            "prediction"
        ]
    )
)

rank_mae = mean_absolute_error(
    ridge_rank_oos[
        TARGET
    ],
    ridge_rank_oos[
        "prediction"
    ]
)

rank_r2 = r2_score(
    ridge_rank_oos[
        TARGET
    ],
    ridge_rank_oos[
        "prediction"
    ]
)

rank_ic = calculate_ic(
    ridge_rank_oos,
    "prediction",
    TARGET
)


rank_prediction_std = (
    ridge_rank_oos
    .groupby(
        "signal_date"
    )[
        "prediction"
    ]
    .std()
    .mean()
)


print(
    "rmse:",
    rank_rmse
)

print(
    "mae:",
    rank_mae
)

print(
    "r2:",
    rank_r2
)

print(
    "mean ic:",
    rank_ic.mean()
)

print(
    "median ic:",
    np.median(
        rank_ic
    )
)

print(
    "ic signals:",
    len(rank_ic)
)

print(
    "prediction std:",
    rank_prediction_std
)

rmse: 0.07860655812671817
mae: 0.054450318519778794
r2: -0.0005774644743394841
mean ic: -0.04866744007019849
median ic: -0.058439375750300115
ic signals: 223
prediction std: 0.0002565534449358558


Cross-sectional rank 표현은 raw feature만 사용했을 때보다 종목 상대순위 정보를 더 잘 표현하는 것으로 보이지만, 아직 안정적인 양의 OOS ranking signal은 만들지 못했다.

Baseline
9 raw asset
+ 8 market
+ 11 macro
= 28

Experiment A
9 raw asset
+ 9 rank
+ 8 market
+ 11 macro
= 37

Experiment B
9 rank asset
+ 8 market
+ 11 macro
= 28

In [20]:
# 06e-16. rank-only feature 설정

RANK_ONLY_FEATURES = (
    RANK_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)


print(
    "baseline:",
    len(BASELINE_FEATURES)
)

print(
    "raw + rank:",
    len(EXPERIMENT_FEATURES)
)

print(
    "rank only:",
    len(RANK_ONLY_FEATURES)
)

baseline: 28
raw + rank: 37
rank only: 28


speaman correlation: 값 자체보다 순위 관계를 보는 상관계수

In [21]:
# 06e-17. raw-rank 상관 확인

raw_rank_correlations = []


for feature in ASSET_FEATURES:

    rank_feature = (
        f"{feature}_rank"
    )

    correlation = (
        supervised_rank_dataset[
            [
                feature,
                rank_feature
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )

    raw_rank_correlations.append(
        {
            "feature":
                feature,
            "rank_feature":
                rank_feature,
            "spearman_corr":
                correlation
        }
    )


raw_rank_correlation_df = (
    pd.DataFrame(
        raw_rank_correlations
    )
)


print(
    raw_rank_correlation_df.to_string(
        index=False
    )
)

                feature                 rank_feature  spearman_corr
              return_1d               return_1d_rank       0.842054
              return_5d               return_5d_rank       0.852621
           momentum_20d            momentum_20d_rank       0.837324
           momentum_60d            momentum_60d_rank       0.836874
         volatility_20d          volatility_20d_rank       0.832770
           drawdown_20d            drawdown_20d_rank       0.851387
     trading_value_ma20      trading_value_ma20_rank       0.754833
trading_value_ratio_20d trading_value_ratio_20d_rank       0.901662
         log_market_cap          log_market_cap_rank       0.949051


In [22]:
# 06e-18. rank-only ridge walk-forward

ridge_rank_only_fold_rows = []
ridge_rank_only_oos_frames = []


for fold in folds:

    train_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["train_end"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            >= fold[
                "validation_start"
            ]
        )
        &
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold[
                "validation_end"
            ]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold[
                "validation_end"
            ]
        )
    )

    test_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            >= fold["test_start"]
        )
        &
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["test_end"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["test_end"]
        )
    )


    train_df = (
        supervised_rank_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_rank_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_rank_dataset.loc[
            test_mask
        ]
        .copy()
    )


    validation_results = (
        evaluate_ridge_alphas(
            train_df,
            validation_df,
            RANK_ONLY_FEATURES
        )
    )


    best_row = (
        validation_results
        .sort_values(
            "rmse"
        )
        .iloc[0]
    )


    best_alpha = float(
        best_row[
            "alpha"
        ]
    )


    train_validation_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["test_start"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["test_start"]
        )
    )


    train_validation_df = (
        supervised_rank_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )


    model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=best_alpha
                )
            )
        ]
    )


    model.fit(
        train_validation_df[
            RANK_ONLY_FEATURES
        ],
        train_validation_df[
            TARGET
        ]
    )


    prediction = model.predict(
        test_df[
            RANK_ONLY_FEATURES
        ]
    )


    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )


    test_result[
        "prediction"
    ] = prediction

    test_result[
        "fold"
    ] = fold["fold"]


    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )


    rmse = np.sqrt(
        mean_squared_error(
            test_result[
                TARGET
            ],
            prediction
        )
    )

    mae = mean_absolute_error(
        test_result[
            TARGET
        ],
        prediction
    )

    r2 = r2_score(
        test_result[
            TARGET
        ],
        prediction
    )


    ridge_rank_only_fold_rows.append(
        {
            "fold":
                fold["fold"],
            "best_alpha":
                best_alpha,
            "validation_rmse":
                best_row["rmse"],
            "rmse":
                rmse,
            "mae":
                mae,
            "r2":
                r2,
            "mean_ic": (
                ic_values.mean()
                if len(ic_values) > 0
                else np.nan
            ),
            "median_ic": (
                np.median(
                    ic_values
                )
                if len(ic_values) > 0
                else np.nan
            ),
            "ic_signals":
                len(ic_values)
        }
    )


    ridge_rank_only_oos_frames.append(
        test_result
    )


    print(
        "fold",
        fold["fold"],
        "done"
    )

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
fold 6 done
fold 7 done
fold 8 done
fold 9 done


In [23]:
# 06e-19. rank-only ridge 결과

ridge_rank_only_fold_results = (
    pd.DataFrame(
        ridge_rank_only_fold_rows
    )
)


ridge_rank_only_oos = (
    pd.concat(
        ridge_rank_only_oos_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


rank_only_rmse = np.sqrt(
    mean_squared_error(
        ridge_rank_only_oos[
            TARGET
        ],
        ridge_rank_only_oos[
            "prediction"
        ]
    )
)

rank_only_mae = mean_absolute_error(
    ridge_rank_only_oos[
        TARGET
    ],
    ridge_rank_only_oos[
        "prediction"
    ]
)

rank_only_r2 = r2_score(
    ridge_rank_only_oos[
        TARGET
    ],
    ridge_rank_only_oos[
        "prediction"
    ]
)

rank_only_ic = calculate_ic(
    ridge_rank_only_oos,
    "prediction",
    TARGET
)

rank_only_prediction_std = (
    ridge_rank_only_oos
    .groupby(
        "signal_date"
    )[
        "prediction"
    ]
    .std()
    .mean()
)


print(
    "rmse:",
    rank_only_rmse
)

print(
    "mae:",
    rank_only_mae
)

print(
    "r2:",
    rank_only_r2
)

print(
    "mean ic:",
    rank_only_ic.mean()
)

print(
    "median ic:",
    np.median(
        rank_only_ic
    )
)

print(
    "ic signals:",
    len(rank_only_ic)
)

print(
    "prediction std:",
    rank_only_prediction_std
)

rmse: 0.07864718308358729
mae: 0.054498397815546454
r2: -0.0016119562811156563
mean ic: 0.012119412787536539
median ic: 0.022040816326530613
ic signals: 223
prediction std: 0.00015737928828910347


동일한 정보를 절대값 대신 상대적 위치로 표현했더니 Ridge가 종목 순서를 더 잘 학습

In [24]:
# 06e-20. feature experiment 비교

feature_experiment_summary = pd.DataFrame(
    [
        {
            "experiment": "raw",
            "rmse": 0.07860736765,
            "r2": -0.00059807,
            "mean_ic": -0.06879395,
            "median_ic": -0.07649460,
            "prediction_std": 0.000241602
        },
        {
            "experiment": "raw_plus_rank",
            "rmse": rank_rmse,
            "r2": rank_r2,
            "mean_ic": rank_ic.mean(),
            "median_ic": np.median(
                rank_ic
            ),
            "prediction_std":
                rank_prediction_std
        },
        {
            "experiment": "rank_only",
            "rmse": rank_only_rmse,
            "r2": rank_only_r2,
            "mean_ic":
                rank_only_ic.mean(),
            "median_ic":
                np.median(
                    rank_only_ic
                ),
            "prediction_std":
                rank_only_prediction_std
        }
    ]
)


print(
    feature_experiment_summary
    .to_string(
        index=False
    )
)

print()

print(
    ridge_rank_only_fold_results[
        [
            "fold",
            "best_alpha",
            "validation_rmse",
            "rmse",
            "r2",
            "mean_ic",
            "median_ic",
            "ic_signals"
        ]
    ]
    .to_string(
        index=False
    )
)

   experiment     rmse        r2   mean_ic  median_ic  prediction_std
          raw 0.078607 -0.000598 -0.068794  -0.076495        0.000242
raw_plus_rank 0.078607 -0.000577 -0.048667  -0.058439        0.000257
    rank_only 0.078647 -0.001612  0.012119   0.022041        0.000157

 fold  best_alpha  validation_rmse     rmse        r2   mean_ic  median_ic  ic_signals
    1   1000000.0         0.065563 0.064497 -0.023631 -0.012898  -0.083313          25
    2     10000.0         0.063463 0.065249 -0.019090 -0.046116  -0.051909          25
    3    100000.0         0.064835 0.074798 -0.017478  0.005654   0.056735          25
    4 100000000.0         0.074470 0.076923 -0.008782  0.065585   0.068619          25
    5 100000000.0         0.076923 0.076069 -0.005062 -0.046512  -0.057047          24
    6 100000000.0         0.076069 0.081828 -0.000332  0.024967   0.064970          25
    7    100000.0         0.081807 0.084058 -0.000533  0.058399   0.051669          24
    8 100000000.0      

In [25]:
# 06e-21. cross-sectional excess target 생성

RELATIVE_TARGET = (
    "target_excess_return"
)


supervised_rank_dataset[
    "target_cross_section_mean"
] = (
    supervised_rank_dataset
    .groupby(
        "signal_date"
    )[TARGET]
    .transform(
        "mean"
    )
)


supervised_rank_dataset[
    RELATIVE_TARGET
] = (
    supervised_rank_dataset[
        TARGET
    ]
    -
    supervised_rank_dataset[
        "target_cross_section_mean"
    ]
)


print(
    supervised_rank_dataset[
        [
            TARGET,
            RELATIVE_TARGET
        ]
    ]
    .describe()
)

       target_return  target_excess_return
count   21779.000000          2.177900e+04
mean        0.001968         -7.773957e-20
std         0.080628          7.185471e-02
min        -0.585680         -5.678377e-01
25%        -0.039127         -3.732524e-02
50%        -0.002803         -5.417537e-03
75%         0.036001          2.979099e-02
max         1.154353          9.819679e-01


In [26]:
# 06e-22. relative target 검증

relative_target_check = (
    supervised_rank_dataset
    .groupby(
        "signal_date"
    )[
        RELATIVE_TARGET
    ]
    .mean()
)


print(
    "max abs mean:",
    relative_target_check
    .abs()
    .max()
)

print(
    "overall mean:",
    supervised_rank_dataset[
        RELATIVE_TARGET
    ]
    .mean()
)

print(
    "nan:",
    supervised_rank_dataset[
        RELATIVE_TARGET
    ]
    .isna()
    .sum()
)

max abs mean: 1.6653345369377347e-17
overall mean: -7.77395708045991e-20
nan: 0


In [27]:
# 06e-23. relative target label 수 확인

target_counts = (
    supervised_rank_dataset
    .groupby(
        "signal_date"
    )[TARGET]
    .count()
)


print(
    target_counts
    .value_counts()
    .sort_index()
)

target_return
49     21
50    415
Name: count, dtype: int64


Experiment C
targer 변경

Experiment B
features = rank-only 28
target   = raw weekly return

Experiment C
features = rank-only 28    ← 그대로
target   = excess return    ← 이것만 변경

In [28]:
# 06e-24. ridge validation 함수 일반화

def evaluate_ridge_alphas_target(
    train_df,
    validation_df,
    features,
    target_column
):

    X_train = train_df[
        features
    ]

    y_train = train_df[
        target_column
    ]

    X_validation = validation_df[
        features
    ]

    y_validation = validation_df[
        target_column
    ]


    results = []


    for alpha in RIDGE_ALPHAS:

        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=alpha
                    )
                )
            ]
        )


        model.fit(
            X_train,
            y_train
        )


        prediction = model.predict(
            X_validation
        )


        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )


        results.append(
            {
                "alpha": alpha,
                "rmse": rmse
            }
        )


    return pd.DataFrame(
        results
    )

In [29]:
# 06e-25. relative-target ridge walk-forward

from sklearn.dummy import DummyRegressor


ridge_relative_fold_rows = []
ridge_relative_oos_frames = []


for fold in folds:

    train_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["train_end"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            >= fold[
                "validation_start"
            ]
        )
        &
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold[
                "validation_end"
            ]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold[
                "validation_end"
            ]
        )
    )

    test_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            >= fold["test_start"]
        )
        &
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["test_end"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["test_end"]
        )
    )


    train_df = (
        supervised_rank_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_rank_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_rank_dataset.loc[
            test_mask
        ]
        .copy()
    )


    validation_results = (
        evaluate_ridge_alphas_target(
            train_df,
            validation_df,
            RANK_ONLY_FEATURES,
            RELATIVE_TARGET
        )
    )


    best_row = (
        validation_results
        .sort_values(
            "rmse"
        )
        .iloc[0]
    )


    best_alpha = float(
        best_row[
            "alpha"
        ]
    )


    train_validation_mask = (
        (
            supervised_rank_dataset[
                "signal_date"
            ]
            < fold["test_start"]
        )
        &
        (
            supervised_rank_dataset[
                "next_execution_date"
            ]
            <= fold["test_start"]
        )
    )


    train_validation_df = (
        supervised_rank_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )


    model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=best_alpha
                )
            )
        ]
    )


    model.fit(
        train_validation_df[
            RANK_ONLY_FEATURES
        ],
        train_validation_df[
            RELATIVE_TARGET
        ]
    )


    prediction = model.predict(
        test_df[
            RANK_ONLY_FEATURES
        ]
    )


    dummy_model = DummyRegressor(
        strategy="mean"
    )

    dummy_model.fit(
        train_validation_df[
            RANK_ONLY_FEATURES
        ],
        train_validation_df[
            RELATIVE_TARGET
        ]
    )

    dummy_prediction = (
        dummy_model.predict(
            test_df[
                RANK_ONLY_FEATURES
            ]
        )
    )


    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET,
                RELATIVE_TARGET
            ]
        ]
        .copy()
    )


    test_result[
        "prediction"
    ] = prediction

    test_result[
        "dummy_prediction"
    ] = dummy_prediction

    test_result[
        "fold"
    ] = fold["fold"]


    ic_values = calculate_ic(
        test_result,
        "prediction",
        RELATIVE_TARGET
    )


    rmse = np.sqrt(
        mean_squared_error(
            test_result[
                RELATIVE_TARGET
            ],
            prediction
        )
    )

    mae = mean_absolute_error(
        test_result[
            RELATIVE_TARGET
        ],
        prediction
    )

    r2 = r2_score(
        test_result[
            RELATIVE_TARGET
        ],
        prediction
    )


    dummy_rmse = np.sqrt(
        mean_squared_error(
            test_result[
                RELATIVE_TARGET
            ],
            dummy_prediction
        )
    )


    ridge_relative_fold_rows.append(
        {
            "fold":
                fold["fold"],
            "best_alpha":
                best_alpha,
            "validation_rmse":
                best_row["rmse"],
            "rmse":
                rmse,
            "mae":
                mae,
            "r2":
                r2,
            "mean_ic": (
                ic_values.mean()
                if len(ic_values) > 0
                else np.nan
            ),
            "median_ic": (
                np.median(
                    ic_values
                )
                if len(ic_values) > 0
                else np.nan
            ),
            "ic_signals":
                len(ic_values),
            "dummy_rmse":
                dummy_rmse,
            "rmse_improvement":
                dummy_rmse
                - rmse,
            "better_than_dummy":
                rmse
                < dummy_rmse
        }
    )


    ridge_relative_oos_frames.append(
        test_result
    )


    print(
        "fold",
        fold["fold"],
        "done"
    )

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
fold 6 done
fold 7 done
fold 8 done
fold 9 done


In [30]:
# 06e-26. relative-target fold 결과

ridge_relative_fold_results = (
    pd.DataFrame(
        ridge_relative_fold_rows
    )
)


print(
    ridge_relative_fold_results
    .to_string(
        index=False
    )
)


print(
    "\nbetter than dummy:",
    ridge_relative_fold_results[
        "better_than_dummy"
    ].sum(),
    "/",
    len(
        ridge_relative_fold_results
    )
)

 fold   best_alpha  validation_rmse     rmse      mae            r2   mean_ic  median_ic  ic_signals  dummy_rmse  rmse_improvement  better_than_dummy
    1 1.000000e+05         0.062327 0.058604 0.043329  3.456571e-05 -0.012844  -0.054694          25    0.058605      1.012867e-06               True
    2 1.000000e+04         0.058603 0.058161 0.041138 -2.040036e-03 -0.045993  -0.057191          25    0.058102     -5.923509e-05              False
    3 1.000000e+08         0.058102 0.068503 0.045239 -9.648533e-08  0.003446   0.054490          25    0.068503     -3.304775e-09              False
    4 1.000000e-04         0.068477 0.070932 0.046571  2.079723e-03  0.023043   0.013782          25    0.071005      7.387425e-05               True
    5 1.000000e-04         0.070932 0.073960 0.050706 -7.371255e-03 -0.069629  -0.067917          24    0.073689     -2.710922e-04              False
    6 1.000000e+08         0.073689 0.074771 0.051533  1.987893e-07  0.026173   0.065354          25

In [31]:
# 06e-27. relative-target oos 결과

ridge_relative_oos = (
    pd.concat(
        ridge_relative_oos_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


relative_rmse = np.sqrt(
    mean_squared_error(
        ridge_relative_oos[
            RELATIVE_TARGET
        ],
        ridge_relative_oos[
            "prediction"
        ]
    )
)

relative_dummy_rmse = np.sqrt(
    mean_squared_error(
        ridge_relative_oos[
            RELATIVE_TARGET
        ],
        ridge_relative_oos[
            "dummy_prediction"
        ]
    )
)

relative_mae = mean_absolute_error(
    ridge_relative_oos[
        RELATIVE_TARGET
    ],
    ridge_relative_oos[
        "prediction"
    ]
)

relative_r2 = r2_score(
    ridge_relative_oos[
        RELATIVE_TARGET
    ],
    ridge_relative_oos[
        "prediction"
    ]
)

relative_ic = calculate_ic(
    ridge_relative_oos,
    "prediction",
    RELATIVE_TARGET
)

relative_prediction_std = (
    ridge_relative_oos
    .groupby(
        "signal_date"
    )[
        "prediction"
    ]
    .std()
    .mean()
)


print(
    "rmse:",
    relative_rmse
)

print(
    "dummy rmse:",
    relative_dummy_rmse
)

print(
    "rmse improvement:",
    relative_dummy_rmse
    - relative_rmse
)

print(
    "mae:",
    relative_mae
)

print(
    "r2:",
    relative_r2
)

print(
    "mean ic:",
    relative_ic.mean()
)

print(
    "median ic:",
    np.median(
        relative_ic
    )
)

print(
    "ic signals:",
    len(relative_ic)
)

print(
    "prediction std:",
    relative_prediction_std
)

rmse: 0.07118897279373791
dummy rmse: 0.07117069774050841
rmse improvement: -1.827505322950229e-05
mae: 0.04914062292168252
r2: -0.0005136214797241134
mean ic: -0.0009867624179716683
median ic: 0.013781512605042016
ic signals: 223
prediction std: 0.0013831697154557444


Experiment 0
raw features + raw target
→ ranking 나쁨

Experiment A
raw + rank + raw target
→ IC 개선, 여전히 음수

Experiment B
rank-only + raw target
→ mean IC +0.0121
→ 6/9 folds positive
→ 현재까지 ranking 관점에서 가장 유망

Experiment C
rank-only + excess target
→ prediction dispersion ↑
→ mean IC ≈ 0
→ Dummy보다 pooled RMSE도 나쁨

결론:
relative target은 채택하지 않음
raw target으로 복귀
rank-only representation 유지

06A–06D
baseline walk-forward evaluation

06E
exploratory model-development walk-forward

In [33]:
# 06e-28. rank-context interaction 생성

CONTEXT_FEATURES = (
    MARKET_FEATURES
    + MACRO_FEATURES
)


interaction_data = {}


for rank_feature in RANK_FEATURES:

    for context_feature in CONTEXT_FEATURES:

        interaction_feature = (
            f"{rank_feature}"
            f"_x_"
            f"{context_feature}"
        )

        interaction_data[
            interaction_feature
        ] = (
            supervised_rank_dataset[
                rank_feature
            ]
            *
            supervised_rank_dataset[
                context_feature
            ]
        )


interaction_features_df = (
    pd.DataFrame(
        interaction_data,
        index=supervised_rank_dataset.index
    )
)


interaction_dataset = (
    pd.concat(
        [
            supervised_rank_dataset,
            interaction_features_df
        ],
        axis=1
    )
    .copy()
)


INTERACTION_FEATURES = list(
    interaction_data.keys()
)


INTERACTION_MODEL_FEATURES = (
    RANK_ONLY_FEATURES
    + INTERACTION_FEATURES
)


print(
    "rank:",
    len(RANK_FEATURES)
)

print(
    "context:",
    len(CONTEXT_FEATURES)
)

print(
    "interactions:",
    len(INTERACTION_FEATURES)
)

print(
    "total:",
    len(INTERACTION_MODEL_FEATURES)
)

rank: 9
context: 19
interactions: 171
total: 199


In [34]:
# 06e-29. interaction 품질 확인

print(
    "rows:",
    len(interaction_dataset)
)

print(
    "signals:",
    interaction_dataset[
        "signal_date"
    ].nunique()
)

print(
    "interaction nan:",
    interaction_dataset[
        INTERACTION_FEATURES
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "interaction inf:",
    np.isinf(
        interaction_dataset[
            INTERACTION_FEATURES
        ]
    )
    .sum()
    .sum()
)

print(
    "features:",
    len(
        INTERACTION_MODEL_FEATURES
    )
)

rows: 21779
signals: 436
interaction nan: 0
interaction inf: 0
features: 199


In [35]:
# 06e-30. ridge walk-forward 함수

def run_ridge_walk_forward(
    data,
    features,
    target_column
):

    fold_rows = []
    oos_frames = []


    for fold in folds:

        train_mask = (
            (
                data["signal_date"]
                < fold["train_end"]
            )
            &
            (
                data["next_execution_date"]
                <= fold["train_end"]
            )
        )

        validation_mask = (
            (
                data["signal_date"]
                >= fold["validation_start"]
            )
            &
            (
                data["signal_date"]
                < fold["validation_end"]
            )
            &
            (
                data["next_execution_date"]
                <= fold["validation_end"]
            )
        )

        test_mask = (
            (
                data["signal_date"]
                >= fold["test_start"]
            )
            &
            (
                data["signal_date"]
                < fold["test_end"]
            )
            &
            (
                data["next_execution_date"]
                <= fold["test_end"]
            )
        )


        train_df = data.loc[
            train_mask
        ].copy()

        validation_df = data.loc[
            validation_mask
        ].copy()

        test_df = data.loc[
            test_mask
        ].copy()


        validation_results = (
            evaluate_ridge_alphas_target(
                train_df,
                validation_df,
                features,
                target_column
            )
        )


        best_row = (
            validation_results
            .sort_values(
                "rmse"
            )
            .iloc[0]
        )

        best_alpha = float(
            best_row[
                "alpha"
            ]
        )


        train_validation_mask = (
            (
                data["signal_date"]
                < fold["test_start"]
            )
            &
            (
                data["next_execution_date"]
                <= fold["test_start"]
            )
        )


        train_validation_df = (
            data.loc[
                train_validation_mask
            ]
            .copy()
        )


        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=best_alpha
                    )
                )
            ]
        )


        model.fit(
            train_validation_df[
                features
            ],
            train_validation_df[
                target_column
            ]
        )


        prediction = model.predict(
            test_df[
                features
            ]
        )


        test_result = (
            test_df[
                [
                    "signal_date",
                    "execution_date",
                    "ticker",
                    "name",
                    target_column
                ]
            ]
            .copy()
        )

        test_result[
            "prediction"
        ] = prediction

        test_result[
            "fold"
        ] = fold["fold"]


        ic_values = calculate_ic(
            test_result,
            "prediction",
            target_column
        )


        rmse = np.sqrt(
            mean_squared_error(
                test_result[
                    target_column
                ],
                prediction
            )
        )

        mae = mean_absolute_error(
            test_result[
                target_column
            ],
            prediction
        )

        r2 = r2_score(
            test_result[
                target_column
            ],
            prediction
        )


        fold_rows.append(
            {
                "fold":
                    fold["fold"],
                "best_alpha":
                    best_alpha,
                "validation_rmse":
                    best_row["rmse"],
                "rmse":
                    rmse,
                "mae":
                    mae,
                "r2":
                    r2,
                "mean_ic": (
                    ic_values.mean()
                    if len(ic_values) > 0
                    else np.nan
                ),
                "median_ic": (
                    np.median(
                        ic_values
                    )
                    if len(ic_values) > 0
                    else np.nan
                ),
                "ic_signals":
                    len(ic_values)
            }
        )


        oos_frames.append(
            test_result
        )


        print(
            "fold",
            fold["fold"],
            "done"
        )


    fold_results = pd.DataFrame(
        fold_rows
    )

    oos = (
        pd.concat(
            oos_frames,
            ignore_index=True
        )
        .sort_values(
            [
                "signal_date",
                "ticker"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return (
        fold_results,
        oos
    )

In [36]:
# 06e-31. interaction ridge 실행

(
    ridge_interaction_fold_results,
    ridge_interaction_oos
) = run_ridge_walk_forward(
    interaction_dataset,
    INTERACTION_MODEL_FEATURES,
    TARGET
)

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
fold 6 done
fold 7 done
fold 8 done
fold 9 done


In [37]:
# 06e-32. interaction ridge 결과

interaction_rmse = np.sqrt(
    mean_squared_error(
        ridge_interaction_oos[
            TARGET
        ],
        ridge_interaction_oos[
            "prediction"
        ]
    )
)

interaction_mae = mean_absolute_error(
    ridge_interaction_oos[
        TARGET
    ],
    ridge_interaction_oos[
        "prediction"
    ]
)

interaction_r2 = r2_score(
    ridge_interaction_oos[
        TARGET
    ],
    ridge_interaction_oos[
        "prediction"
    ]
)

interaction_ic = calculate_ic(
    ridge_interaction_oos,
    "prediction",
    TARGET
)

interaction_prediction_std = (
    ridge_interaction_oos
    .groupby(
        "signal_date"
    )[
        "prediction"
    ]
    .std()
    .mean()
)


print(
    ridge_interaction_fold_results
    .to_string(
        index=False
    )
)

print()

print(
    "rmse:",
    interaction_rmse
)

print(
    "mae:",
    interaction_mae
)

print(
    "r2:",
    interaction_r2
)

print(
    "mean ic:",
    interaction_ic.mean()
)

print(
    "median ic:",
    np.median(
        interaction_ic
    )
)

print(
    "ic signals:",
    len(
        interaction_ic
    )
)

print(
    "prediction std:",
    interaction_prediction_std
)

 fold  best_alpha  validation_rmse     rmse      mae        r2   mean_ic  median_ic  ic_signals
    1 100000000.0         0.065564 0.064547 0.047620 -0.025218 -0.014907  -0.002449          25
    2     10000.0         0.064034 0.065340 0.047617 -0.021933  0.042509   0.014070          25
    3    100000.0         0.064727 0.074877 0.049238 -0.019631  0.017516  -0.021849          25
    4 100000000.0         0.074470 0.076923 0.050796 -0.008780  0.001097  -0.011861          25
    5   1000000.0         0.076921 0.076087 0.051550 -0.005550 -0.042434  -0.028331          24
    6 100000000.0         0.076069 0.081828 0.057120 -0.000331  0.033526   0.012149          25
    7   1000000.0         0.081819 0.084022 0.061990  0.000327  0.082490   0.077551          24
    8    100000.0         0.083991 0.079491 0.056500 -0.050466  0.043270   0.036062          25
    9 100000000.0         0.079276 0.099643 0.068624 -0.031804  0.027246   0.051333          25

rmse: 0.07869075278821391
mae: 0.054540

현재 06E 결과
→ rank representation: 유효 가능성 있음
→ market/macro interaction: ranking 추가 개선
→ raw return magnitude 예측: 여전히 약함

다음
→ 새로운 stock-specific characteristics 추가
   beta / idio vol / reversal 등

그 후
→ Ridge로 빠른 검증
→ 의미 있으면 RF/XGB에도 적용